# 01_eda: Exploratory Data Analysis
Understand the loan data, find leakage columns, and check class imbalance (REQ-01).

In [0]:
%run ./00_config

## Step 1: List files in the data folder
The brief gives us a folder path but not the file names. Before writing any code against the data,
we list what is actually inside the folder. This way we build on facts, not assumptions.

In [0]:
# WHAT: List every file inside the Lending Club sample folder.
# WHY:  The brief only gives the folder path. We need the real file names
#       (and their sizes) before we can read the data correctly.

files = dbutils.fs.ls("/databricks-datasets/lending-club-loan-stats/")
display(files)

**Result:** The folder contains 2 files:
- `LoanStats_2018Q2.csv` (~67 MB): the raw loan records we will use.
- `lendingClubData.txt` (~1 KB): a short description file.

**Why it matters:** The file name says *2018 Q2*, so the data may only cover loans issued in April–June 2018.
If so, many loans may still be "Current" (outcome not known yet), which will affect how we build the target
and how we split train/test by time. We will confirm this with real numbers when we load the data.

## Step 2: Read the dataset's description file
Before loading the 67 MB CSV, we read the small description file that ships with it.
It may tell us the data's source, time period, or known quirks.

In [0]:
# WHAT: Print the contents of the small description file.
# WHY:  It may explain where the data comes from and which time period it covers.
#       That context guides how we read and split the data later.

print(dbutils.fs.head("/databricks-datasets/lending-club-loan-stats/lendingClubData.txt"))

**Result:** The description says the data comes from LendingClub's public loan exports and includes
each loan's *current status* (Current, Late, Fully Paid, etc.) and its *latest payment information*.

**Why it matters:**
1. `loan_status` has more than two values, so we must decide how to turn it into a yes/no target (some loans are still "Current").
2. "Latest payment information" means some columns describe what happened **after** the loan was issued.
   These are **data leakage** risks and must be found and removed in EDA.
3. It is not yet clear whether the file covers only 2018 Q2 or all loans up to 2018 Q2. We will check `issue_d`.

## Step 3: Peek at the raw CSV text
Real-world CSV exports often contain extra lines (titles, notes) above the header or below the data.
We look at the first characters of the raw file so we know exactly how to read it into Spark.

In [0]:
# WHAT: Print the first 1,500 characters of the raw loan CSV as plain text.
# WHY:  To check the file's real structure (is line 1 the header? any extra title lines?)
#       before telling Spark how to read it.

print(dbutils.fs.head("/databricks-datasets/lending-club-loan-stats/LoanStats_2018Q2.csv", 1500))

## Step 4: Load the data and count rows and columns

In [0]:

# Read the CSV into a Spark DataFrame
raw_df = (spark.read
          .option("header", True)       # line 1 = column names (checked in Step 3)
          .option("inferSchema", True)  # let Spark guess each column's type
          .csv(RAW_CSV_PATH))

# Count rows and columns. This is our starting size.
print(f"Rows:    {raw_df.count():,}")
print(f"Columns: {len(raw_df.columns)}")

Loaded **130,776 rows** and **145 columns**. We will only keep the useful, safe columns.